# 🌱 Python para análisis estadístico en Biología

## Notebook para completar

Este notebook reúne los contenidos esenciales para una sesión introductoria de análisis estadístico con Python, con ejemplos orientados a biología vegetal.

### Objetivos

Al finalizar, deberíamos ser capaces de:

- resumir un conjunto de datos mediante estadística descriptiva;
- visualizar distribuciones y comparar grupos;
- formular e interpretar hipótesis estadísticas;
- aplicar un **t-test** para comparar dos grupos;
- aplicar un **ANOVA de una vía** para comparar tres o más grupos;
- entender por qué se utilizan comparaciones **post-hoc**;
- mencionar alternativas no paramétricas;
- ajustar modelos no lineales mediante `scipy.optimize.curve_fit`;
- interpretar parámetros de modelos logístico y Gompertz;
- integrar varias de estas herramientas en un problema biológico completo.

> **Importante:** la estadística no consiste solamente en ejecutar una función.  
> Antes de aplicar un test debemos entender qué pregunta queremos responder y cuáles son sus supuestos.

# 1. Importación de librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats
from scipy.optimize import curve_fit

# 2. Estadística descriptiva

La estadística descriptiva resume las principales características de los datos.

Algunas medidas frecuentes son:

- media;
- mediana;
- mínimo y máximo;
- varianza;
- desviación estándar;
- percentiles.

La **media** describe el centro de los datos, mientras que la **desviación estándar** describe cuánto se dispersan alrededor de la media.

In [ ]:
alturas = np.array([18.2, 19.1, 17.8, 20.3, 21.0, 18.9, 19.7, 20.1, 18.5, 19.4])

## Visualización rápida de una distribución

# 3. De la descripción a la inferencia

Supongamos que queremos comparar dos tratamientos.

Tenemos:

- **Control**
- **Tratamiento**

La estadística descriptiva puede mostrarnos que sus medias son distintas, pero necesitamos preguntarnos:

> ¿La diferencia observada podría explicarse simplemente por variabilidad aleatoria?

Aquí comienza la **inferencia estadística**.

## Hipótesis nula y alternativa

Para comparar dos medias:

$$
H_0: \mu_{\text{control}} = \mu_{\text{tratamiento}}
$$

$$
H_1: \mu_{\text{control}} \neq \mu_{\text{tratamiento}}
$$

La hipótesis nula representa el escenario de referencia: **no existe diferencia entre las medias poblacionales**.

## p-value

El p-value puede interpretarse como:

> La probabilidad de observar un resultado tan extremo como el obtenido, o más extremo, **suponiendo que la hipótesis nula sea verdadera**.

No significa:

* la probabilidad de que la hipótesis nula sea cierta;
* la probabilidad de que nuestro resultado sea correcto;
* el tamaño del efecto biológico.

Utilizaremos convencionalmente:

$$
\alpha = 0.05
$$

Si:

$$
p < 0.05
$$

diremos que existe evidencia suficiente para **rechazar** la hipótesis nula.

Si:

$$
p \geq 0.05
$$

diremos que **no existe evidencia suficiente para rechazarla**.

No debemos decir que hemos demostrado que los grupos son iguales.

# 4. Comparación de dos grupos: t-test

In [ ]:
control = np.array([18.2, 19.1, 17.8, 20.3, 18.9, 19.5, 18.7, 19.0])
tratamiento = np.array([21.0, 22.1, 20.7, 23.0, 21.8, 22.4, 21.5, 22.0])

## t-test de muestras independientes

Usaremos el **t-test de Welch**:

```python
stats.ttest_ind(grupo1, grupo2, equal_var=False)
```

Es una versión del t-test que no requiere asumir igualdad de varianzas.

## Tamaño del efecto

Una diferencia puede ser estadísticamente significativa pero biológicamente pequeña.

Podemos calcular una diferencia absoluta entre medias:

Una interpretación razonable debería mencionar tanto:

- evidencia estadística;
- magnitud de la diferencia.

# 5. t-test pareado

Un test pareado se utiliza cuando las observaciones están relacionadas.

Ejemplo:

- medir las mismas plantas antes y después de un tratamiento;
- comparar dos condiciones sobre el mismo individuo.

En este caso **no** debemos usar un t-test independiente.

In [ ]:
antes = np.array([10.2, 11.1, 9.8, 10.5, 11.0, 10.8])
despues = np.array([11.4, 12.0, 10.7, 11.2, 11.9, 11.5])

# 6. La unidad experimental y la pseudorreplicación

Supongamos:

- 3 plantas control;
- 3 plantas tratamiento;
- 10 hojas medidas por planta.

Aunque tengamos 30 mediciones de hojas por grupo, **no necesariamente tenemos 30 réplicas biológicas independientes**.

Las hojas de una misma planta comparten el mismo organismo, ambiente y contexto experimental.

Tratar todas las hojas como réplicas independientes puede producir una falsa sensación de tamaño muestral grande y p-values artificialmente pequeños.

> Antes de aplicar un test, debemos identificar correctamente cuál es la unidad experimental.

# 7. Comparación de tres o más grupos: ANOVA de una vía

Supongamos ahora que tenemos:

* Control
* Fertilizante A
* Fertilizante B

El ANOVA plantea:

$$
H_0: \mu_C = \mu_A = \mu_B
$$

La hipótesis alternativa establece que **al menos una media difiere**.

In [ ]:
control = np.array([18.1, 19.0, 18.5, 19.3, 18.8, 19.1])
fert_a = np.array([20.1, 20.5, 19.8, 21.0, 20.7, 20.3])
fert_b = np.array([23.0, 22.4, 23.5, 22.9, 23.2, 22.7])

## ¿Qué significa un ANOVA significativo?

Si:

$$
p < 0.05
$$

podemos decir:

> Existe evidencia de que no todas las medias son iguales.

Pero **NO** podemos concluir directamente:

* que todos los grupos sean diferentes;
* qué pares específicos difieren.

Para eso necesitamos comparaciones **post-hoc**.

# 8. Comparaciones post-hoc

Una estrategia habitual es utilizar **Tukey HSD**.

SciPy incluye:

```python
stats.tukey_hsd(...)
```

La salida contiene comparaciones pareadas entre los grupos.

La lógica es:

1. ANOVA responde si existe evidencia global de diferencias.
2. Tukey ayuda a identificar qué pares presentan diferencias.

# 9. Métodos paramétricos y no paramétricos

En términos introductorios:

### Métodos paramétricos

Utilizan modelos que dependen de parámetros poblacionales y generalmente realizan ciertos supuestos sobre la distribución de los errores.

Ejemplos:

- t-test;
- ANOVA.

### Métodos no paramétricos

Realizan menos supuestos distribucionales y frecuentemente trabajan con rangos.

Algunas equivalencias útiles:

| Problema | Paramétrico | No paramétrico |
|---|---|---|
| Dos grupos independientes | t-test | Mann–Whitney U |
| Dos mediciones pareadas | t-test pareado | Wilcoxon |
| Tres o más grupos | ANOVA | Kruskal–Wallis |

No debemos simplificar esto como:

> "normal → paramétrico; no normal → no paramétrico"

La elección depende también del diseño experimental, independencia, tamaño muestral, outliers y pregunta científica.

In [ ]:
grupo_a = np.array([2, 3, 3, 4, 4, 5])
grupo_b = np.array([6, 7, 8, 8, 9, 10])

# 10. Ajuste de modelos no lineales

En otros problemas no queremos comparar grupos, sino describir cómo una variable cambia en función de otra.

Por ejemplo:

> ¿Cómo cambia la biomasa de una planta a través del tiempo?

Podemos proponer una función:

$$
y = f(t, \theta)
$$

donde:

* $t$ es el tiempo;
* $y$ es la variable observada;
* $\theta$ representa los parámetros del modelo.

`scipy.optimize.curve_fit` estima los parámetros que mejor ajustan la curva a los datos.

# 11. Modelo logístico

Una forma habitual del modelo logístico es:

$$
y(t)=\frac{K}{1+e^{-r(t-t_0)}}
$$

donde:

* $K$: valor máximo o asíntota;
* $r$: tasa de crecimiento;
* $t_0$: tiempo del punto de inflexión.

In [ ]:
tiempo = np.array([0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30], dtype=float)
biomasa = np.array([1.8, 2.3, 3.5, 5.7, 9.5, 15.0, 21.8, 27.0, 30.4, 32.1, 33.0])

## Interpretación

Si obtenemos aproximadamente:

* $K \approx 34$: el modelo predice que la biomasa se aproxima a ese valor máximo;
* $r$: describe la rapidez de la transición;
* $t_0$: indica aproximadamente cuándo ocurre el máximo crecimiento instantáneo.

En el modelo logístico estándar:

$$
y(t_0)=\frac{K}{2}
$$

# 12. Modelo de Gompertz

Otra función sigmoidea frecuente en biología es Gompertz:

$$
y(t)=K e^{-e^{-r(t-t_0)}}
$$

A diferencia de la logística, la curva de Gompertz es **asimétrica**.

# 13. ¿Cómo comparar ajustes de manera simple?

Una opción introductoria consiste en comparar los residuos.

Para cada observación:

$$
e_i = y_i - \hat{y}_i
$$

y luego calcular el **error cuadrático medio**:

$$
MSE = \frac{1}{n}\sum_i (y_i-\hat{y}_i)^2
$$

Un MSE menor indica un ajuste más cercano a los datos observados, aunque no necesariamente implica que el modelo sea biológicamente mejor.


# 14. Ejercicio Completo

## Situación

Se estudia el efecto de tres tratamientos sobre plantas:

- Control
- Fertilizante A
- Fertilizante B

Después de 30 días se mide la biomasa final.

Además, para el tratamiento B se registra la biomasa a través del tiempo.

### Objetivos

1. Resumir los datos.
2. Visualizar los tratamientos.
3. Aplicar un ANOVA.
4. Si corresponde, realizar Tukey HSD.
5. Interpretar la magnitud de las diferencias.
6. Ajustar un modelo logístico al crecimiento del tratamiento B.
7. Interpretar sus parámetros.

In [ ]:
control = np.array([21.4, 20.8, 22.1, 21.7, 20.9, 21.3, 22.0, 21.1])
fert_a = np.array([24.8, 25.1, 24.2, 25.5, 24.9, 25.3, 24.6, 25.0])
fert_b = np.array([30.1, 29.4, 30.8, 31.0, 29.9, 30.5, 30.2, 29.7])

## Parte 1 — Estadística descriptiva

## Parte 2 — Visualización

## Parte 3 — ANOVA

### Hipótesis

$$
H_0: \mu_C = \mu_A = \mu_B
$$

$$
H_1: \text{al menos una media difiere}
$$

## Parte 4 — Tukey HSD

## Parte 5 — Magnitud de las diferencias

### Interpretación

Una conclusión adecuada combinaría:

- el resultado del ANOVA;
- las comparaciones post-hoc;
- la magnitud de las diferencias.

No basta con escribir únicamente:

> p < 0.05

También debemos explicar **qué grupos difieren** y **cuánto difieren**.

## Parte 6 — Crecimiento temporal del tratamiento B

In [ ]:
dias = np.array([0, 4, 8, 12, 16, 20, 24, 28, 32], dtype=float)
biomasa_b = np.array([1.6, 2.7, 5.4, 10.2, 17.8, 24.3, 28.4, 30.0, 30.7])

## Parte 7 — Ajuste logístico

## Parte 8 — Interpretación biológica

Los parámetros estimados pueden interpretarse aproximadamente como:

- **K:** biomasa máxima o asíntota predicha;
- **r:** rapidez del crecimiento;
- **t0:** momento aproximado del punto de inflexión.

Por lo tanto, el análisis no solamente permite decir si los tratamientos difieren, sino también describir cuantitativamente la dinámica de crecimiento.

# 15. Resumen final

## Antes de aplicar un test

Preguntar:

1. ¿Cuál es mi pregunta biológica?
2. ¿Cuál es mi unidad experimental?
3. ¿Las observaciones son independientes?
4. ¿Tengo dos grupos, más de dos grupos o una relación continua?
5. ¿Qué supuestos estoy haciendo?

## Herramientas vistas

| Pregunta | Herramienta |
|---|---|
| ¿Cómo son mis datos? | media, mediana, SD, histogramas, boxplots |
| ¿Difieren dos grupos independientes? | Welch t-test |
| ¿Difieren dos mediciones relacionadas? | t-test pareado |
| ¿Difieren tres o más grupos? | ANOVA |
| ¿Qué grupos difieren? | Tukey HSD |
| ¿Alternativa no paramétrica para dos grupos? | Mann–Whitney |
| ¿Alternativa no paramétrica para 3+ grupos? | Kruskal–Wallis |
| ¿Cómo describo crecimiento sigmoideo? | Logístico / Gompertz |
| ¿Cómo estimo parámetros? | `curve_fit` |

> La estadística no consiste en buscar un `p < 0.05`, sino en responder una pregunta científica utilizando un análisis apropiado y una interpretación cuidadosa.